# Vision Transformer PTQ investigation on CIFAR-10

This notebook loads the public ImageNet-pretrained ViT-B/16 weights, trains a CIFAR-10 classification head when no saved checkpoint exists, and compares the original FP32 model, built-in TFLite dynamic-range PTQ, and custom per-output-channel INT8 weight PTQ. It also investigates parameter ranks, quantized tensor/value coverage, TFLite graph dtypes, and parameter-memory reduction.

Both accuracy comparisons are weight-only with FP32 activations. Custom activation ranges are calibrated and reported separately but are not applied during inference. No fake-quantization API is used.

In [1]:
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory
import gc
import json
import sys

import keras
import keras_hub
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.lite.python import schema_py_generated as schema_fb

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.tflite_metrics import inspect_tflite
from src.models import CIFAR10_CLASS_NAMES
from src.models.transformer_based import (
    VIT_BASE_PATCH16_224_IMAGENET_PRESET,
    build_vit_cifar10_classifier,
    build_vit_image_preprocessor,
)
from src.quantization.custom_quantization import custom_ptq, dequantize_tensor

np.random.seed(42)
tf.random.set_seed(42)
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
print(f"KerasHub: {keras_hub.__version__}")

TensorFlow: 2.20.0
Keras: 3.14.1
KerasHub: 0.29.1


/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

The default first run trains only the new CIFAR-10 head while keeping the pretrained ViT backbone frozen. Set `FREEZE_BACKBONE=False` for full fine-tuning. Set `MAX_TRAIN_SAMPLES=None` to use all 50,000 training images.

In [2]:
PRESET = VIT_BASE_PATCH16_224_IMAGENET_PRESET
IMAGE_SIZE = (224, 224)
NUM_CLASSES = 10
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "vit_cifar10_ptq"
MODEL_PATH = OUTPUT_DIR / "vit_cifar10_fp32.keras"
BUILTIN_MODEL_PATH = OUTPUT_DIR / "vit_cifar10_builtin_dynamic_int8.tflite"

MAX_TRAIN_SAMPLES = 10_000
NUM_EVALUATION_SAMPLES = None
NUM_CALIBRATION_SAMPLES = 100
TRAIN_BATCH_SIZE = 16
EVALUATION_BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 1e-3  # Linear-probe learning rate for the new head.
FREEZE_BACKBONE = True
FORCE_RETRAIN = False
FORCE_RECONVERT_BUILTIN = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saved classifier: {MODEL_PATH}")
print(f"Built-in PTQ model: {BUILTIN_MODEL_PATH}")

Saved classifier: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_cifar10_ptq/vit_cifar10_fp32.keras
Built-in PTQ model: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_cifar10_ptq/vit_cifar10_builtin_dynamic_int8.tflite


## 2. Load CIFAR-10 and matching ViT preprocessing

CIFAR-10 is used so the ViT result can be compared with the existing ResNet-18 study on the same ten-class task. The preset preprocessor performs the required resize and normalization outside the model being quantized.

In [3]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
train_labels = train_labels.reshape(-1).astype(np.int32)
test_labels = test_labels.reshape(-1).astype(np.int32)
if MAX_TRAIN_SAMPLES is not None:
    rng = np.random.default_rng(42)
    selected = rng.permutation(len(train_images))[:MAX_TRAIN_SAMPLES]
    train_images = train_images[selected]
    train_labels = train_labels[selected]
if NUM_EVALUATION_SAMPLES is not None:
    test_images = test_images[:NUM_EVALUATION_SAMPLES]
    test_labels = test_labels[:NUM_EVALUATION_SAMPLES]

preprocessor = build_vit_image_preprocessor()

def preprocess_images(images):
    return preprocessor(images)

print(f"Training images: {len(train_images):,}")
print(f"Evaluation images: {len(test_images):,}")
print(f"Preprocessed shape: {preprocess_images(train_images[:1]).shape}")

100%|██████████| 909/909 [00:00<00:00, 1.15MB/s]


100%|██████████| 1.74k/1.74k [00:00<00:00, 1.55MB/s]

Training images: 10,000
Evaluation images: 10,000
Preprocessed shape: (1, 224, 224, 3)


## 3. Load existing weights and obtain a CIFAR-10 classifier

If the cached CIFAR-10 checkpoint exists, it is loaded without training. Otherwise, `build_vit_cifar10_classifier()` loads the existing ImageNet-pretrained ViT-B/16 backbone, attaches a new ten-class Dense head, trains it, and caches the resulting model.

In [4]:
if MODEL_PATH.exists() and not FORCE_RETRAIN:
    model = keras.models.load_model(MODEL_PATH, compile=False)
    print(f"Loaded trained checkpoint: {MODEL_PATH}")
else:
    model = build_vit_cifar10_classifier(
        num_classes=NUM_CLASSES,
        freeze_backbone=FREEZE_BACKBONE,
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    train_dataset = (
        tf.data.Dataset.from_tensor_slices((train_images, train_labels))
        .shuffle(min(len(train_labels), 10_000), seed=42)
        .batch(TRAIN_BATCH_SIZE)
        .map(lambda images, labels: (preprocess_images(images), labels),
             num_parallel_calls=tf.data.AUTOTUNE)
        .prefetch(tf.data.AUTOTUNE)
    )
    model.fit(train_dataset, epochs=EPOCHS)
    model.save(MODEL_PATH)
    print(f"Saved trained classifier: {MODEL_PATH}")

model.summary(expand_nested=True)

100%|██████████| 328M/328M [00:09<00:00, 36.0MB/s] 


Epoch 1/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 1876s 3s/step - accuracy: 0.9289 - loss: 0.2365
Epoch 2/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 2193s 4s/step - accuracy: 0.9710 - loss: 0.0932
Epoch 3/3
625/625 ━━━━━━━━━━━━━━━━━━━━ 1897s 3s/step - accuracy: 0.9790 - loss: 0.0670
Saved trained classifier: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_cifar10_ptq/vit_cifar10_fp32.keras


Model: "vit_cifar10_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                  ┃ Output Shape                       ┃             Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ images (InputLayer)                           │ (None, 224, 224, 3)                │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ vi_t_backbone (ViTBackbone)                   │ (None, 197, 768)                   │          85,798,656 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│    └ images (InputLayer)                      │ (None, 224, 224, 3)                │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│    └ vit_patching_and_embedding               │ (None, 197, 768)                   │             742,656 │
│ (ViTPatchingAndEmbedding)                     │                                    │                     │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│    └ vit_encoder (ViTEncoder)                 │ (None, 197, 768)                   │          85,056,000 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ get_item (GetItem)                            │ (None, 768)                        │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ output_dropout (Dropout)                      │ (None, 768)                        │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ predictions (Dense)                           │ (None, 10)                         │               7,690 │
└───────────────────────────────────────────────┴────────────────────────────────────┴─────────────────────┘

 Total params: 85,821,728 (327.38 MB)

 Trainable params: 7,690 (30.04 KB)

 Non-trainable params: 85,798,656 (327.30 MB)

 Optimizer params: 15,382 (60.09 KB)

## 4. Apply custom per-channel INT8 weight PTQ

Custom PTQ quantizes floating parameter tensors with rank `>= 2`. This includes the patch-projection kernel, positional embeddings, attention projections, MLP kernels, and classifier kernel. Rank-1 biases and LayerNorm parameters remain FP32. The quantized weights are reconstructed into a Keras model for FP32 inference; calibrated activation parameters are not applied.

In [5]:
calibration_count = min(NUM_CALIBRATION_SAMPLES, len(train_images))
calibration_samples = [
    preprocess_images(train_images[index : index + 1]).numpy()
    for index in range(calibration_count)
]
custom_results = custom_ptq(
    model,
    representative_samples=calibration_samples,
    quantize_min_rank=2,
    per_channel=True,
)
weight_result = custom_results["weights"]
activation_result = custom_results.get("activations")

custom_model = keras.models.load_model(MODEL_PATH, compile=False)
quantized_tensor_iter = iter(weight_result.tensors)
reconstructed_weights = []
for weight in model.weights:
    weight_array = weight.numpy()
    should_quantize = (
        np.issubdtype(weight_array.dtype, np.floating)
        and weight_array.ndim >= 2
    )
    if should_quantize:
        reconstructed_weights.append(
            dequantize_tensor(next(quantized_tensor_iter)).astype(weight_array.dtype)
        )
    else:
        reconstructed_weights.append(weight_array)
if list(quantized_tensor_iter):
    raise RuntimeError("Not all custom quantized tensors were consumed.")
custom_model.set_weights(reconstructed_weights)
del reconstructed_weights
gc.collect()

pd.DataFrame([{
    "quantized_weight_tensors": len(weight_result.tensors),
    "calibrated_activation_tensors": len(activation_result.ranges),
    "fp32_parameter_mib": weight_result.fp32_size_bytes / 1024**2,
    "custom_parameter_mib": weight_result.quantized_size_bytes / 1024**2,
    "compression_ratio": weight_result.compression_ratio,
    "memory_reduction_percent": weight_result.memory_reduction_percent,
}])

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


,quantized_weight_tensors,calibrated_activation_tensors,fp32_parameter_mib,custom_parameter_mib,compression_ratio,memory_reduction_percent
0,112,3,327.325233,82.101601,3.986831,74.917424


## 5. Convert with built-in TFLite dynamic-range PTQ

The preprocessed image tensor is the model input. Dynamic-range PTQ stores supported constant weights as INT8 while executing activations in FP32, making it the closest built-in comparison to the current custom W8 evaluation.

In [6]:
def build_fixed_vit_model(classifier):
    images = keras.Input(shape=(*IMAGE_SIZE, 3), dtype=tf.float32, name="images")
    logits = classifier(images, training=False)
    return keras.Model(images, logits, name="vit_cifar10_fixed")

if FORCE_RECONVERT_BUILTIN or not BUILTIN_MODEL_PATH.exists():
    fixed_model = build_fixed_vit_model(model)
    with TemporaryDirectory(dir=OUTPUT_DIR) as saved_model_dir:
        fixed_model.export(saved_model_dir, format="tf_saved_model")
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        BUILTIN_MODEL_PATH.write_bytes(converter.convert())
    del fixed_model
    gc.collect()

builtin_storage = inspect_tflite(BUILTIN_MODEL_PATH)
print(f"Built-in model: {BUILTIN_MODEL_PATH}")
print(f"Serialized size: {BUILTIN_MODEL_PATH.stat().st_size / 1024**2:.2f} MiB")
builtin_storage

INFO:tensorflow:Assets written to: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_cifar10_ptq/tmpooofrx4d/assets


INFO:tensorflow:Assets written to: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_cifar10_ptq/tmpooofrx4d/assets


Saved artifact at '/Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_cifar10_ptq/tmpooofrx4d'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  4887605520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4887605136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4887608016: TensorSpec(shape=(1, 197), dtype=tf.int32, name=None)
  4887606288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4887606864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4887607248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4887604944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4887604560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4887607440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  4887607056: TensorSpec(shape=(), dtype=tf.resource, name

W0000 00:00:1786742316.117968 13069844 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1786742316.118090 13069844 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-08-14 23:18:36.120197: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_cifar10_ptq/tmpooofrx4d
2026-08-14 23:18:36.123480: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-08-14 23:18:36.123487: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_cifar10_ptq/tmpooofrx4d
I0000 00:00:1786742316.154532 13069844 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-08-14 23:18:36.160197: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-08-14 23:18:36.444007: I tensorflow/cc/saved_model/loader.cc:220] Ru

Built-in model: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_cifar10_ptq/vit_cifar10_builtin_dynamic_int8.tflite
Serialized size: 83.75 MiB


/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


{'input_dtype': 'float32',
 'output_dtype': 'float32',
 'tensor_dtype_counts': {'float32': 723, 'int32': 19, 'int8': 74},
 'weight_storage_bytes': 86629056,
 'activation_tensor_storage_bytes': 385455640,
 'largest_activation_tensor_bytes': 2420736}

## 6. Tensor rank, value-count, and memory investigation

In [7]:
def count_tensors_by_rank(keras_model, quantize_min_rank=2):
    rows = []
    for weight in keras_model.weights:
        values = weight.numpy()
        rows.append({
            "tensor": getattr(weight, "path", weight.name),
            "rank": values.ndim,
            "shape": tuple(values.shape),
            "number_of_values": values.size,
            "quantized_by_custom_ptq": (
                np.issubdtype(values.dtype, np.floating)
                and values.ndim >= quantize_min_rank
            ),
        })
    detail = pd.DataFrame(rows)
    summary = (
        detail.groupby("rank", as_index=False)
        .agg(
            total_tensors=("tensor", "count"),
            total_values=("number_of_values", "sum"),
            custom_quantized_tensors=("quantized_by_custom_ptq", "sum"),
        )
        .sort_values("rank")
    )
    summary["custom_unquantized_tensors"] = (
        summary["total_tensors"] - summary["custom_quantized_tensors"]
    )
    return summary, detail

rank_summary, individual_tensor_ranks = count_tensors_by_rank(model)
rank_summary

,rank,total_tensors,total_values,custom_quantized_tensors,custom_unquantized_tensors
0,1,88,94474,0,88
1,2,62,56809728,62,0
2,3,49,28312320,49,0
3,4,1,589824,1,0


In [8]:
def inspect_tflite_constants(model_path):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    details = interpreter.get_tensor_details()
    details_by_index = {int(detail["index"]): detail for detail in details}
    content = Path(model_path).read_bytes()
    flatbuffer = schema_fb.Model.GetRootAsModel(content, 0)
    subgraph = flatbuffer.Subgraphs(0)
    rows = []
    for tensor_index in range(subgraph.TensorsLength()):
        tensor = subgraph.Tensors(tensor_index)
        buffer = flatbuffer.Buffers(tensor.Buffer())
        detail = details_by_index.get(tensor_index)
        if buffer.DataLength() == 0 or detail is None:
            continue
        rows.append({
            "name": detail["name"],
            "dtype": np.dtype(detail["dtype"]).name,
            "values": int(np.prod(detail["shape"], dtype=np.int64)),
            "storage_bytes": int(buffer.DataLength()),
        })
    return details, pd.DataFrame(rows)

builtin_details, builtin_constants = inspect_tflite_constants(BUILTIN_MODEL_PATH)
builtin_int8_constants = builtin_constants[builtin_constants["dtype"] == "int8"]
original_parameter_values = int(sum(w.numpy().size for w in model.weights))
original_parameter_bytes = int(sum(w.numpy().nbytes for w in model.weights))
custom_int8_values = int(sum(t.values.size for t in weight_result.tensors))
builtin_parameter_bytes = int(builtin_storage["weight_storage_bytes"])

tensor_analysis_table = pd.DataFrame([
    {"model": "Original FP32 ViT", "parameter_or_constant_tensors": len(model.weights), "int8_tensors": 0, "total_values": original_parameter_values, "int8_values": 0, "graph_tensors": None},
    {"model": "Custom per-channel W8", "parameter_or_constant_tensors": len(model.weights), "int8_tensors": len(weight_result.tensors), "total_values": original_parameter_values, "int8_values": custom_int8_values, "graph_tensors": None},
    {"model": "Built-in dynamic-range TFLite", "parameter_or_constant_tensors": len(builtin_constants), "int8_tensors": len(builtin_int8_constants), "total_values": int(builtin_constants["values"].sum()), "int8_values": int(builtin_int8_constants["values"].sum()), "graph_tensors": len(builtin_details)},
])
memory_analysis_table = pd.DataFrame([
    {"model": "Original FP32 ViT", "raw_parameter_bytes": original_parameter_bytes},
    {"model": "Custom per-channel W8", "raw_parameter_bytes": int(weight_result.quantized_size_bytes)},
    {"model": "Built-in dynamic-range TFLite", "raw_parameter_bytes": builtin_parameter_bytes},
])
memory_analysis_table["raw_parameter_mib"] = memory_analysis_table["raw_parameter_bytes"] / 1024**2
memory_analysis_table["compression_ratio_vs_fp32"] = original_parameter_bytes / memory_analysis_table["raw_parameter_bytes"]
memory_analysis_table["memory_reduction_percent"] = (1 - memory_analysis_table["raw_parameter_bytes"] / original_parameter_bytes) * 100
builtin_graph_dtype_counts = dict(Counter(np.dtype(d["dtype"]).name for d in builtin_details))

display(tensor_analysis_table)
display(pd.DataFrame([builtin_graph_dtype_counts], index=["built-in graph"]))
display(builtin_constants.groupby("dtype", as_index=False).agg(constant_tensors=("name", "count"), constant_values=("values", "sum"), storage_bytes=("storage_bytes", "sum")))
display(memory_analysis_table)

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,model,parameter_or_constant_tensors,int8_tensors,total_values,int8_values,graph_tensors
0,Original FP32 ViT,200,0,85806346,0,NaN
1,Custom per-channel W8,200,112,85806346,85711872,NaN
2,Built-in dynamic-range TFLite,217,74,85806386,85532160,816.0


,float32,int32,int8
built-in graph,723,19,74


,dtype,constant_tensors,constant_values,storage_bytes
0,float32,128,274188,1096752
1,int32,15,38,152
2,int8,74,85532160,85532160


,model,raw_parameter_bytes,raw_parameter_mib,compression_ratio_vs_fp32,memory_reduction_percent
0,Original FP32 ViT,343225384,327.325233,1.000000,0.000000
1,Custom per-channel W8,86089768,82.101601,3.986831,74.917424
2,Built-in dynamic-range TFLite,86629056,82.615906,3.962012,74.760300


## 7. Shared prediction helpers and one-image check

In [9]:
def predict_keras_labels(keras_model, raw_images, batch_size=EVALUATION_BATCH_SIZE):
    predictions = []
    for start in range(0, len(raw_images), batch_size):
        images = preprocess_images(raw_images[start : start + batch_size])
        logits = keras_model(images, training=False).numpy()
        predictions.append(np.argmax(logits, axis=1))
    return np.concatenate(predictions)

def predict_tflite_labels(model_path, raw_images, batch_size=EVALUATION_BATCH_SIZE):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    predictions = []
    for start in range(0, len(raw_images), batch_size):
        images = preprocess_images(raw_images[start : start + batch_size]).numpy()
        input_detail = interpreter.get_input_details()[0]
        interpreter.resize_tensor_input(input_detail["index"], images.shape, strict=False)
        interpreter.allocate_tensors()
        input_detail = interpreter.get_input_details()[0]
        values = images.astype(input_detail["dtype"])
        if input_detail["dtype"] != np.float32:
            scale, zero_point = input_detail["quantization"]
            limits = np.iinfo(input_detail["dtype"])
            values = np.clip(np.rint(images / scale + zero_point), limits.min, limits.max).astype(input_detail["dtype"])
        interpreter.set_tensor(input_detail["index"], values)
        interpreter.invoke()
        logits = interpreter.get_tensor(interpreter.get_output_details()[0]["index"])
        predictions.append(np.argmax(logits, axis=1))
    return np.concatenate(predictions)

sample_label = int(test_labels[0])
sample_predictions = {
    "Original FP32 ViT": int(predict_keras_labels(model, test_images[:1], 1)[0]),
    "Built-in dynamic-range PTQ": int(predict_tflite_labels(BUILTIN_MODEL_PATH, test_images[:1], 1)[0]),
    "Custom per-channel W8": int(predict_keras_labels(custom_model, test_images[:1], 1)[0]),
}
pd.DataFrame([{
    "model": name,
    "true_class": CIFAR10_CLASS_NAMES[sample_label],
    "predicted_class": CIFAR10_CLASS_NAMES[prediction],
    "correct": prediction == sample_label,
} for name, prediction in sample_predictions.items()])

/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,model,true_class,predicted_class,correct
0,Original FP32 ViT,cat,cat,True
1,Built-in dynamic-range PTQ,cat,cat,True
2,Custom per-channel W8,cat,cat,True


## 8. Accuracy comparison

All rows use the same CIFAR-10 test images and matching ViT preprocessing.

In [12]:
fp32_predictions = predict_keras_labels(model, test_images)
builtin_predictions = predict_tflite_labels(BUILTIN_MODEL_PATH, test_images)
custom_predictions = predict_keras_labels(custom_model, test_images)
fp32_accuracy = float(np.mean(fp32_predictions == test_labels))
builtin_accuracy = float(np.mean(builtin_predictions == test_labels))
custom_accuracy = float(np.mean(custom_predictions == test_labels))

accuracy_table = pd.DataFrame([
    {"model": "Original FP32 ViT", "quantization": "FP32 baseline", "correct_predictions": int(np.sum(fp32_predictions == test_labels)), "test_images": len(test_labels), "accuracy_percent": fp32_accuracy * 100, "drop_from_fp32_percentage_points": 0.0},
    {"model": "Built-in dynamic-range PTQ", "quantization": "supported W8 constants; FP32 activations", "correct_predictions": int(np.sum(builtin_predictions == test_labels)), "test_images": len(test_labels), "accuracy_percent": builtin_accuracy * 100, "drop_from_fp32_percentage_points": (fp32_accuracy - builtin_accuracy) * 100},
    {"model": "Custom per-channel W8", "quantization": "rank >= 2 W8; FP32 Keras activations", "correct_predictions": int(np.sum(custom_predictions == test_labels)), "test_images": len(test_labels), "accuracy_percent": custom_accuracy * 100, "drop_from_fp32_percentage_points": (fp32_accuracy - custom_accuracy) * 100},
])
accuracy_table

,model,quantization,correct_predictions,test_images,accuracy_percent,drop_from_fp32_percentage_points
0,Original FP32 ViT,FP32 baseline,9573,10000,95.73,0.00
1,Built-in dynamic-range PTQ,supported W8 constants; FP32 activations,9578,10000,95.78,-0.05
2,Custom per-channel W8,rank >= 2 W8; FP32 Keras activations,9579,10000,95.79,-0.06


## 9. Save results

In [13]:
evaluation_images = test_images[:1000]
evaluation_labels = test_labels[:1000]

print("Evaluating FP32...")
fp32_predictions = predict_keras_labels(model, evaluation_images)

print("Evaluating built-in PTQ...")
builtin_predictions = predict_tflite_labels(
    BUILTIN_MODEL_PATH,
    evaluation_images,
)

print("Evaluating custom PTQ...")
custom_predictions = predict_keras_labels(
    custom_model,
    evaluation_images,
)

fp32_accuracy = float(np.mean(fp32_predictions == evaluation_labels))
builtin_accuracy = float(np.mean(builtin_predictions == evaluation_labels))
custom_accuracy = float(np.mean(custom_predictions == evaluation_labels))

accuracy_table = pd.DataFrame([
    {
        "model": "Original FP32 ViT",
        "quantization": "FP32 baseline",
        "correct_predictions": int(np.sum(fp32_predictions == evaluation_labels)),
        "test_images": len(evaluation_labels),
        "accuracy_percent": fp32_accuracy * 100,
        "drop_from_fp32_percentage_points": 0.0,
    },
    {
        "model": "Built-in dynamic-range PTQ",
        "quantization": "supported W8 constants; FP32 activations",
        "correct_predictions": int(np.sum(builtin_predictions == evaluation_labels)),
        "test_images": len(evaluation_labels),
        "accuracy_percent": builtin_accuracy * 100,
        "drop_from_fp32_percentage_points": (
            fp32_accuracy - builtin_accuracy
        ) * 100,
    },
    {
        "model": "Custom per-channel W8",
        "quantization": "rank >= 2 W8; FP32 Keras activations",
        "correct_predictions": int(np.sum(custom_predictions == evaluation_labels)),
        "test_images": len(evaluation_labels),
        "accuracy_percent": custom_accuracy * 100,
        "drop_from_fp32_percentage_points": (
            fp32_accuracy - custom_accuracy
        ) * 100,
    },
])

accuracy_table

Evaluating FP32...


KeyboardInterrupt: 

In [14]:
accuracy_path = OUTPUT_DIR / "ptq_accuracy_comparison.csv"
rank_path = OUTPUT_DIR / "parameter_rank_summary.csv"
tensor_path = OUTPUT_DIR / "parameter_tensor_details.csv"
memory_path = OUTPUT_DIR / "parameter_memory_comparison.csv"
results_path = OUTPUT_DIR / "results.json"
accuracy_table.to_csv(accuracy_path, index=False)
rank_summary.to_csv(rank_path, index=False)
individual_tensor_ranks.to_csv(tensor_path, index=False)
memory_analysis_table.to_csv(memory_path, index=False)
results_path.write_text(json.dumps({
    "dataset": "CIFAR-10 test",
    "preset": PRESET,
    "training_images": int(len(train_labels)),
    "evaluation_images": int(len(test_labels)),
    "calibration_images": calibration_count,
    "backbone_frozen_during_training": FREEZE_BACKBONE,
    "fp32_accuracy_percent": fp32_accuracy * 100,
    "builtin_dynamic_ptq_accuracy_percent": builtin_accuracy * 100,
    "custom_weight_ptq_accuracy_percent": custom_accuracy * 100,
    "custom_activation_calibration_applied_during_inference": False,
}, indent=2) + "\n", encoding="utf-8")
print(f"Saved results under: {OUTPUT_DIR}")

Saved results under: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_cifar10_ptq


## Final Vision Transformer PTQ accuracy comparison (%)

In [15]:
final_accuracy_table = accuracy_table.copy()
final_accuracy_table["accuracy_percent"] = final_accuracy_table["accuracy_percent"].map(lambda value: f"{value:.2f}%")
final_accuracy_table["drop_from_fp32_percentage_points"] = final_accuracy_table["drop_from_fp32_percentage_points"].map(lambda value: f"{value:.2f}")
final_accuracy_table

,model,quantization,correct_predictions,test_images,accuracy_percent,drop_from_fp32_percentage_points
0,Original FP32 ViT,FP32 baseline,9573,10000,95.73%,0.00
1,Built-in dynamic-range PTQ,supported W8 constants; FP32 activations,9578,10000,95.78%,-0.05
2,Custom per-channel W8,rank >= 2 W8; FP32 Keras activations,9579,10000,95.79%,-0.06
